<a href="https://colab.research.google.com/github/c-marq/cap4767-data-mining/blob/main/demos/week02_demo_retail_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 Demo — Forecasting U.S. Retail Sales
## Presenting Time Series Results to a Board of Directors

**CAP4767 Data Mining with Python** | Miami Dade College — Kendall Campus

---

### Tonight's Scenario

You've just been hired as the **lead data analyst** for a national restaurant and food services group. The company operates 1,200 locations across the U.S. The CFO has asked your team to build a **36-month sales forecast** that will drive three board-level decisions:

1. **Capital allocation** — Should the company open 80 new locations in 2024–2025, or hold cash?
2. **Supply chain contracts** — Should they lock in a 3-year food supply contract at a fixed rate, or keep buying quarter-to-quarter?
3. **Staffing model** — Which months need surge hiring, and which months can run lean?

The board doesn't want to see code. They want to see **one chart, two numbers, and a recommendation.** Your job tonight is to learn what those numbers mean well enough to defend them in a boardroom.

---

### What We'll Cover (~90 Minutes)

| Block | Time | Topic |
|-------|------|-------|
| 1 | 10 min | Data exploration — what does this series look like? |
| 2 | 5 min | The log-transform — one line of code that changes everything |
| 3 | 5 min | The smart train/test split |
| 4 | 15 min | SARIMAX — build, forecast, evaluate |
| 5 | 15 min | Prophet — build, forecast, evaluate |
| 6 | 25 min | **Deep Dive: What the metrics MEAN** — RMSE, R², p-values, AIC |
| 7 | 15 min | Model comparison, board recommendation, and discussion |


---
## Block 1: Setup & Exploration (~10 min)


In [ ]:
# ============================================================
# Setup — Run this cell first. Do not modify.
# ============================================================
!pip install -q pmdarima prophet


In [ ]:
# ============================================================
# Imports & Data Loading — Run this cell. Do not modify.
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
import logging

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from pmdarima import auto_arima
from prophet import Prophet
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["figure.dpi"] = 100

# Load U.S. Advance Retail Sales: Food Services & Drinking Places
# Source: FRED series RSAFSNA (millions of dollars, not seasonally adjusted)
data_url = "https://raw.githubusercontent.com/c-marq/cap4767-data-mining/main/data/rsafsna_retail_sales.csv"
df = pd.read_csv(data_url, parse_dates=["DATE"], index_col="DATE")
ts = df["retail_sales"].asfreq("MS")

print(f"Dataset: {df.shape[0]} monthly observations")
print(f"Date range: {ts.index[0].strftime('%b %Y')} to {ts.index[-1].strftime('%b %Y')}")
print(f"Unit: Millions of USD (not seasonally adjusted)")
df.head()


### 1a. Summary Statistics


In [ ]:
# 1a. Summary statistics
print(f"Mean:  ${ts.mean():,.0f}M")
print(f"Min:   ${ts.min():,.0f}M")
print(f"Max:   ${ts.max():,.0f}M")
print(f"Range: ${ts.max() - ts.min():,.0f}M")
print(f"\nThe data spans ~${(ts.max() - ts.min())/1000:,.0f} billion — from ${ts.min()/1000:,.0f}B to ${ts.max()/1000:,.0f}B.")
print(f"When we get an RMSE later, remember this range — it's your yardstick.")


### 1b. Time Series Plot


In [ ]:
# 1b. Full time series plot
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(ts.index, ts.values, color="steelblue", linewidth=1.2)
ax.set_title("U.S. Retail Sales: Food Services & Drinking Places (1992–2026)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Sales (Millions USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:,.0f}B'))
ax.grid(True, alpha=0.3)

# Annotate key features
ax.annotate("COVID-19\nshutdowns", xy=(pd.Timestamp("2020-04-01"), 401556),
            xytext=(pd.Timestamp("2016-01-01"), 420000),
            arrowprops=dict(arrowstyle="->", color="red", lw=1.5),
            fontsize=10, color="red", fontweight="bold")

ax.annotate("Dec seasonal\npeaks ↑", xy=(pd.Timestamp("2024-12-01"), 786673),
            xytext=(pd.Timestamp("2023-01-01"), 820000),
            arrowprops=dict(arrowstyle="->", color="green", lw=1.5),
            fontsize=10, color="green", fontweight="bold")

plt.tight_layout()
plt.show()


<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DISCUSSION — What Do You See?</strong><br>
  Three things to identify before we run any tests:
  <ol>
    <li><strong>Trend:</strong> Is the series going up, down, or flat over 30+ years?</li>
    <li><strong>Seasonality:</strong> Do you see a repeating annual pattern? Which month is always the peak?</li>
    <li><strong>Anomaly:</strong> There's one massive disruption. When is it and what caused it?</li>
  </ol>
  One more thing — look at the <em>shape</em> of the growth. Is the gap between the peaks in 2020 and 2024 the same size as the gap between 1995 and 1999? The growth is <strong>accelerating</strong> — that's going to matter in a minute.
</div>


### 1c. ADF Stationarity Test


In [ ]:
# 1c. ADF Stationarity Test
adf_result = adfuller(ts.dropna(), autolag="AIC")

print("AUGMENTED DICKEY-FULLER TEST")
print("=" * 50)
print(f"ADF Statistic:  {adf_result[0]:.4f}")
print(f"P-value:        {adf_result[1]:.6f}")
print()
if adf_result[1] < 0.05:
    print("✅ RESULT: Data IS stationary (p < 0.05)")
else:
    print("⚠️  RESULT: Data is NOT stationary (p ≥ 0.05)")
    print(f"   P-value of {adf_result[1]:.4f} means we CANNOT reject the null hypothesis.")
    print(f"   The data has a strong trend — the mean is NOT stable over time.")
    print(f"   SARIMAX will need differencing. But we also have another tool...")


In [ ]:
# 1d. Seasonal Decomposition
decomposition = seasonal_decompose(ts, model="additive", period=12)
fig = decomposition.plot()
fig.set_size_inches(13, 9)
fig.suptitle("Seasonal Decomposition — Additive, Period=12 (Monthly)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 READING THE DECOMPOSITION</strong><br>
  Four panels, four stories:
  <ul>
    <li><strong>Observed:</strong> The raw data — everything combined.</li>
    <li><strong>Trend:</strong> The long-term direction. Steady climb from ~$150B in 1992 to ~$700B+ by 2025, with the COVID dip around 2020.</li>
    <li><strong>Seasonal:</strong> The repeating 12-month pattern. <strong>December is always the peak</strong> (holiday dining) and <strong>January/February are always the trough</strong> (post-holiday drop).</li>
    <li><strong>Residual:</strong> What's left after removing trend and seasonality. The huge spike around 2020? That's COVID — neither trend nor seasonality explains it.</li>
  </ul>
</div>


---
## Block 2: The Log-Transform (~5 min)

<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 WHY LOG-TRANSFORM?</strong><br>
  Look at the raw plot again. In 1995, a December peak was ~$20B above the year's average. In 2024, a December peak is ~$80B above the year's average. The <em>seasonal swings are getting bigger as the trend grows</em>. That's called <strong>multiplicative seasonality</strong> — the seasonal effect is proportional to the level.<br><br>
  SARIMAX works best when seasonal effects are a <strong>constant size</strong> (additive). The log-transform fixes this by converting multiplication into addition. When you take the log of exponential growth, you get a straight line. When you take the log of growing seasonal swings, they become constant-sized swings.<br><br>
  This is <strong>standard practice</strong> in industry for economic time series — GDP, sales revenue, stock prices, population — anything that grows proportionally over time.
</div>


In [ ]:
# 2a. Apply the log-transform — ONE LINE OF CODE
ts_log = np.log(ts)

# Show the before/after
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ts.index, ts.values, color="steelblue", linewidth=1)
axes[0].set_title("BEFORE: Raw Sales (Accelerating Growth)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Sales (Millions USD)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:,.0f}B'))
axes[0].grid(True, alpha=0.3)

axes[1].plot(ts_log.index, ts_log.values, color="darkorange", linewidth=1)
axes[1].set_title("AFTER: Log-Transformed (Stabilized Growth)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Log(Sales)")
axes[1].grid(True, alpha=0.3)

fig.suptitle("The Log-Transform: One Line of Code, Massive Impact",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Left panel:  The seasonal peaks get TALLER over time (multiplicative).")
print("Right panel: The seasonal peaks are roughly the SAME HEIGHT throughout (additive).")
print("\nSARIMAX can model the right panel much more effectively.")
print("\nThe code: ts_log = np.log(ts)")
print("To convert back: np.exp(forecast) — we'll do this after forecasting.")


<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #7D6608;">⚠️ THE WORKFLOW</strong><br>
  <ol>
    <li>Take the log of the raw data: <code>ts_log = np.log(ts)</code></li>
    <li>Train and forecast in log-space (the model works with the transformed numbers)</li>
    <li>Convert the forecast back to real dollars: <code>np.exp(forecast)</code></li>
    <li>Evaluate RMSE and R² on the <strong>real dollar values</strong> — always report in the units the business understands</li>
  </ol>
</div>


---
## Block 3: Train/Test Split (~5 min)

<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 WHY 2010–2022?</strong><br>
  We're using the most recent 13 years of training data rather than the full 30+ years. Restaurant spending patterns in the 1990s represent a fundamentally different economy — different price levels, different consumer behavior, different industry structure. Training on modern data gives both models a more relevant growth trajectory.<br><br>
  COVID (2020) is <strong>inside the training window</strong> — the models see the crash and the recovery, so they know it happened. The test set (2023–2025) is clean post-recovery data — a fair evaluation.
</div>


In [ ]:
# 3. Train/Test Split
# Train on log-transformed data: Jan 2010 – Dec 2022 (156 months)
# Test on log-transformed data:  Jan 2023 – Dec 2025 (36 months)
# Actuals kept in original scale for evaluation

train_log = ts_log["2010-01":"2022-12"]
test_log = ts_log["2023-01":"2025-12"]
test_actual = ts["2023-01":"2025-12"]    # Real dollars for evaluation
train_actual = ts["2010-01":"2022-12"]   # Real dollars for plotting

print(f"Train: {len(train_log)} months ({train_log.index[0].strftime('%b %Y')} – {train_log.index[-1].strftime('%b %Y')})")
print(f"Test:  {len(test_log)} months ({test_log.index[0].strftime('%b %Y')} – {test_log.index[-1].strftime('%b %Y')})")
print(f"\nTest mean (real dollars): ${test_actual.mean():,.0f}M")

# Visualize the split (in real dollars for readability)
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train_actual.index, train_actual, label="Train (2010–2022)", color="steelblue")
ax.plot(test_actual.index, test_actual, label="Test (2023–2025)", color="darkorange", linewidth=2)
ax.axvline(x=test_actual.index[0], color="red", linestyle="--", alpha=0.7, label="Split Point")
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-01"),
           alpha=0.15, color="red", label="COVID period (in training)")
ax.set_title("Train/Test Split — U.S. Food Services Sales", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Sales (Millions USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:,.0f}B'))
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## Block 4: SARIMAX Forecasting (~15 min)


In [ ]:
# 4a. auto_arima on log-transformed training data
print("Running auto_arima on log-transformed data (this may take 1–2 minutes)...")
print("It's testing hundreds of (p,d,q)(P,D,Q) combinations.\n")

auto_model = auto_arima(
    train_log,
    seasonal=True,
    m=12,
    suppress_warnings=True,
    error_action="ignore",
    trace=False,
    stepwise=True
)

order = auto_model.order
seasonal_order = auto_model.seasonal_order

print("=" * 50)
print("AUTO_ARIMA RESULTS (on log-transformed data)")
print("=" * 50)
print(f"Best non-seasonal order (p,d,q): {order}")
print(f"Best seasonal order (P,D,Q,m):   {seasonal_order}")
print(f"AIC: {auto_model.aic():.2f}")
print()
print("--- What Each Parameter Means ---")
print(f"  p={order[0]} → Uses {order[0]} recent month(s) as predictors (autoregressive lags)")
print(f"  d={order[1]} → Differences the data {order[1]} time(s) to remove trend")
print(f"  q={order[2]} → Uses {order[2]} recent forecast error(s) for self-correction")
print(f"  P={seasonal_order[0]} → Uses {seasonal_order[0]} past same-month value(s) as seasonal predictors")
print(f"  D={seasonal_order[1]} → Seasonal differencing {seasonal_order[1]} time(s)")
print(f"  Q={seasonal_order[2]} → Uses {seasonal_order[2]} past seasonal forecast error(s)")
print(f"  m={seasonal_order[3]} → Seasonal cycle length = {seasonal_order[3]} months")


In [ ]:
# 4b. Fit SARIMAX on log data, forecast, back-transform to dollars
model = SARIMAX(
    train_log,
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarimax_result = model.fit(disp=False)

# Forecast in log-space
sarimax_forecast_log = sarimax_result.forecast(steps=len(test_log))

# Back-transform to real dollars
sarimax_forecast = np.exp(sarimax_forecast_log)

# Evaluate on real dollars
sarimax_rmse = np.sqrt(mean_squared_error(test_actual, sarimax_forecast))
sarimax_r2 = r2_score(test_actual, sarimax_forecast)

print("=" * 50)
print("SARIMAX EVALUATION ON TEST SET (2023–2025)")
print("=" * 50)
print(f"RMSE:  ${sarimax_rmse:,.0f} million")
print(f"R²:    {sarimax_r2:.4f}  ({sarimax_r2*100:.1f}%)")
print()
print(f"In plain English:")
print(f"  → The forecast is off by ~${sarimax_rmse:,.0f}M per month on average")
print(f"  → The model explains {sarimax_r2*100:.1f}% of the variation in the test data")
print(f"  → Relative to the test mean (${test_actual.mean():,.0f}M),")
print(f"    RMSE is {sarimax_rmse/test_actual.mean()*100:.1f}% of typical monthly sales")


In [ ]:
# 4c. SARIMAX model summary — statistical detail
print(sarimax_result.summary())


<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 READING THE SARIMAX SUMMARY TABLE</strong><br>
  The model summary shows a coefficient table with four key columns:
  <ul>
    <li><strong>coef</strong> — the learned weight for each parameter. Larger absolute values = stronger influence.</li>
    <li><strong>std err</strong> — how uncertain the model is about that coefficient. Smaller = more confident.</li>
    <li><strong>z</strong> — the z-statistic (coef ÷ std err). Farther from zero = more statistically significant.</li>
    <li><strong>P>|z|</strong> — <strong>this is the p-value.</strong> If P>|z| < 0.05, that parameter is statistically significant — the model is confident it's a real pattern, not noise.</li>
  </ul>
  Notice how the <strong>non-seasonal parameters</strong> (ar.L1, ar.L2, ma.L1) now have significant p-values — that's the log-transform at work. On the raw data, none of these were significant. The transform gave the model a clean enough signal to find real patterns.
</div>


In [ ]:
# 4d. Plot SARIMAX forecast vs actuals (in real dollars)
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train_actual.index[-60:], train_actual[-60:],
        label="Train (last 5 yrs)", color="steelblue", alpha=0.6)
ax.plot(test_actual.index, test_actual,
        label="Actual (2023–2025)", color="darkorange", linewidth=2)
ax.plot(test_actual.index, sarimax_forecast,
        label="SARIMAX Forecast", linestyle="--", color="green", linewidth=2)
ax.axvline(x=test_actual.index[0], color="red", linestyle="--", alpha=0.5)
ax.set_title(f"SARIMAX Forecast — U.S. Retail Food Services Sales\n"
             f"RMSE = ${sarimax_rmse:,.0f}M  |  R² = {sarimax_r2:.4f} ({sarimax_r2*100:.1f}%)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Sales (Millions USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:,.0f}B'))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## Block 5: Prophet Forecasting (~15 min)

<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 WHY PROPHET ON LOG DATA TOO?</strong><br>
  To make the comparison fair, we give Prophet the same log-transformed data. We use <strong>additive</strong> seasonality here — because the log already converted the multiplicative relationship into an additive one. We then back-transform Prophet's forecast to real dollars with <code>np.exp()</code>, exactly like we did for SARIMAX.
</div>


In [ ]:
# 5a. Prepare Prophet format & fit on log data
train_prophet = pd.DataFrame({"ds": train_log.index, "y": train_log.values})

prophet_model = Prophet(
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10.0,
    seasonality_mode="additive",
    changepoint_range=0.9,
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)
prophet_model.fit(train_prophet)

# Forecast in log-space
future = prophet_model.make_future_dataframe(periods=len(test_log), freq="MS")
forecast_df = prophet_model.predict(future)
prophet_forecast_log = forecast_df["yhat"].iloc[-len(test_log):].values

# Back-transform to real dollars
prophet_forecast = np.exp(prophet_forecast_log)

# Evaluate on real dollars
prophet_rmse = np.sqrt(mean_squared_error(test_actual, prophet_forecast))
prophet_r2 = r2_score(test_actual, prophet_forecast)

print("=" * 50)
print("PROPHET EVALUATION ON TEST SET (2023–2025)")
print("=" * 50)
print(f"RMSE:  ${prophet_rmse:,.0f} million")
print(f"R²:    {prophet_r2:.4f}  ({prophet_r2*100:.1f}%)")
print()
print(f"In plain English:")
print(f"  → The forecast is off by ~${prophet_rmse:,.0f}M per month on average")
print(f"  → The model explains {prophet_r2*100:.1f}% of the variation in the test data")
print(f"  → Relative to the test mean (${test_actual.mean():,.0f}M),")
print(f"    RMSE is {prophet_rmse/test_actual.mean()*100:.1f}% of typical monthly sales")


In [ ]:
# 5b. Plot Prophet forecast vs actuals (in real dollars)
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train_actual.index[-60:], train_actual[-60:],
        label="Train (last 5 yrs)", color="steelblue", alpha=0.6)
ax.plot(test_actual.index, test_actual,
        label="Actual (2023–2025)", color="darkorange", linewidth=2)
ax.plot(test_actual.index, prophet_forecast,
        label="Prophet Forecast", linestyle="--", color="purple", linewidth=2)
ax.axvline(x=test_actual.index[0], color="red", linestyle="--", alpha=0.5)
ax.set_title(f"Prophet Forecast — U.S. Retail Food Services Sales\n"
             f"RMSE = ${prophet_rmse:,.0f}M  |  R² = {prophet_r2:.4f} ({prophet_r2*100:.1f}%)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Sales (Millions USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:,.0f}B'))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 5c. Prophet component plots (in log-space — this is how Prophet sees the data)
fig = prophet_model.plot_components(forecast_df)
plt.suptitle("Prophet Components (Log-Space) — Trend + Yearly Seasonality",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Note: These components are in log-space.")
print("The trend shows log(sales) over time — a straight-ish line means steady % growth.")
print("The seasonal panel shows which months are above/below the trend.")


<div style="background-color: #FADBD8; border-left: 5px solid #E74C3C; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #922B21;">🛑 STOP AND COMPARE</strong><br>
  Look at the SARIMAX and Prophet results side by side:
  <ul>
    <li>SARIMAX R² should be <strong>positive and meaningful</strong> — the model is capturing real patterns</li>
    <li>Prophet R² is likely <strong>negative</strong> — worse than guessing the average</li>
    <li>Both models have negative bias (they over-predict), but SARIMAX's bias is smaller</li>
  </ul>
  We'll dig into <em>why</em> in Block 6. For now, note the contrast — same data, same log-transform, very different results.
</div>


---
## Block 6: Deep Dive — What the Metrics MEAN (~25 min)

This is the core of tonight's lesson. We're going to take each metric apart, understand it conceptually, and learn how to communicate it to a non-technical audience like a board of directors.


### 6a. RMSE — Root Mean Squared Error

<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 THE CONCEPT</strong><br>
  RMSE answers one question: <strong>"On average, how far off is our forecast from reality?"</strong><br><br>
  It's measured in the <em>same units as the data</em> — in this case, millions of dollars. That's what makes it powerful for business communication. You don't say "our RMSE is 0.03" — you say <strong>"our forecast is typically within $23,000 million of actual sales."</strong>
</div>


In [ ]:
# 6a. RMSE — Build it from scratch to understand it

# Step 1: Calculate individual errors (actual - predicted)
errors_sarimax = test_actual.values - sarimax_forecast.values
errors_prophet = test_actual.values - prophet_forecast

# Step 2: Square each error (makes all positive, penalizes large errors more)
squared_errors_sarimax = errors_sarimax ** 2

# Step 3: Take the mean of the squared errors
mse_sarimax = squared_errors_sarimax.mean()

# Step 4: Take the square root to get back to original units
rmse_manual = np.sqrt(mse_sarimax)

print("RMSE — STEP BY STEP (SARIMAX)")
print("=" * 55)
print(f"\nStep 1 — Errors (first 6 months):")
for i in range(6):
    actual = test_actual.values[i]
    pred = sarimax_forecast.values[i]
    err = actual - pred
    print(f"  {test_actual.index[i].strftime('%b %Y')}: "
          f"Actual ${actual:,.0f}M − Predicted ${pred:,.0f}M = Error ${err:+,.0f}M")

print(f"\nStep 2 — Square each error (penalizes big misses more)")
print(f"Step 3 — Average the squared errors → MSE = {mse_sarimax:,.0f}")
print(f"Step 4 — Square root → RMSE = ${rmse_manual:,.0f}M")
print(f"\n✅ Matches sklearn: ${sarimax_rmse:,.0f}M")

print(f"\n" + "=" * 55)
print(f"BOARD-READY INTERPRETATION:")
print(f"=" * 55)
pct_of_mean = sarimax_rmse / test_actual.mean() * 100
print(f'"Our SARIMAX model forecasts monthly food service sales')
print(f"  within ~${sarimax_rmse:,.0f} million of actual — that's about {pct_of_mean:.1f}%")
print(f'  of a typical month\'s sales."')


<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #7D6608;">⚠️ RMSE — WHAT THE BOARD ASKS</strong><br>
  The CFO will not ask "what is the RMSE?" The CFO will ask: <strong>"How much should I budget for forecast error?"</strong><br><br>
  Your answer translates RMSE into dollars:<br>
  <em>"Based on our model's accuracy, we recommend budgeting a ±3% buffer on each month's revenue projection. For a projected $700M month, that means planning for a range of roughly $677M to $723M."</em>
</div>


### 6b. R² — Coefficient of Determination

<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 THE CONCEPT</strong><br>
  R² answers: <strong>"How much of what's happening in the data does our model actually capture?"</strong><br><br>
  It compares your model against the <strong>dumbest possible baseline</strong> — predicting the mean value for every single month. If R² = 0.76, your model captures 76% of the patterns. If R² = 0, your model is no better than guessing the average. If R² is <em>negative</em>, your model is <strong>worse</strong> than guessing the average — that's a model you throw away.
</div>


In [ ]:
# 6b. R² — Build it from scratch

mean_value = test_actual.mean()

# Total variance: how much the actual data moves around its own mean
ss_total = np.sum((test_actual.values - mean_value) ** 2)

# Residual variance: how much our forecasts miss by
ss_residual_sarimax = np.sum((test_actual.values - sarimax_forecast.values) ** 2)
ss_residual_prophet = np.sum((test_actual.values - prophet_forecast) ** 2)

r2_manual_sarimax = 1 - (ss_residual_sarimax / ss_total)
r2_manual_prophet = 1 - (ss_residual_prophet / ss_total)

print("R² — HOW IT WORKS")
print("=" * 55)
print(f"\nBaseline: The 'dumb' model predicts ${mean_value:,.0f}M every month.")
print(f"\nTotal variance in actual data (SS_total):       {ss_total:,.0f}")
print(f"SARIMAX residual variance (SS_residual):         {ss_residual_sarimax:,.0f}")
print(f"Prophet residual variance (SS_residual):         {ss_residual_prophet:,.0f}")
print()
print(f"R² = 1 − (SS_residual / SS_total)")
print(f"  SARIMAX: 1 − ({ss_residual_sarimax:,.0f} / {ss_total:,.0f}) = {r2_manual_sarimax:.4f} ({r2_manual_sarimax*100:.1f}%)")
print(f"  Prophet: 1 − ({ss_residual_prophet:,.0f} / {ss_total:,.0f}) = {r2_manual_prophet:.4f} ({r2_manual_prophet*100:.1f}%)")
print(f"\n✅ Matches sklearn — SARIMAX: {sarimax_r2:.4f} | Prophet: {prophet_r2:.4f}")


In [ ]:
# 6b (continued). R² Visual — Model vs Baseline
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, name, forecast, r2, color in [
    (axes[0], "SARIMAX", sarimax_forecast, sarimax_r2, "green"),
    (axes[1], "Prophet", prophet_forecast, prophet_r2, "purple")
]:
    ax.plot(test_actual.index, test_actual, label="Actual", color="darkorange", linewidth=2)
    forecast_vals = forecast.values if hasattr(forecast, 'values') else forecast
    ax.plot(test_actual.index, forecast_vals, label=f"{name} Forecast",
            linestyle="--", color=color, linewidth=2)
    ax.axhline(y=mean_value, color="gray", linestyle=":", alpha=0.8,
               label=f"Mean (${mean_value:,.0f}M)")
    r2_label = f"{r2*100:.1f}%" if r2 > 0 else f"{r2:.2f} (worse than mean)"
    ax.set_title(f"{name} — R² = {r2_label}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Month")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:,.0f}B'))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Sales (Millions USD)")
fig.suptitle("R² Comparison: How Much Better Than Guessing the Average?",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("The gray dotted line is the \"dumb\" baseline — always predicting the mean.")
print("R² measures how much closer the colored dashed line is to the orange vs. the gray.")
print(f"\nSARIMAX tracks the orange line closely → R² = {sarimax_r2:.4f} ({sarimax_r2*100:.1f}% of patterns captured)")
print(f"Prophet diverges from the orange line → R² = {prophet_r2:.4f} (worse than guessing the mean)")


<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ R² — WHAT TO TELL THE BOARD</strong><br>
  <strong>Don't say:</strong> "The R-squared is 0.76."<br>
  <strong>Say:</strong> "Our model captures 76% of the month-to-month variation in sales. The remaining 24% is unpredictable variation — things like weather events, viral social media trends, or local economic shifts that no model can anticipate."<br><br>
  <strong>If R² is negative</strong> (like Prophet here), say: "This model performed worse than a simple average — it's not reliable for this forecast and should not be used for decision-making."
</div>


### 6c. P-Values — Statistical Significance of Model Parameters

<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 THE CONCEPT</strong><br>
  A p-value answers: <strong>"Is this pattern real, or could it be random noise?"</strong><br><br>
  Every coefficient in the SARIMAX model has a p-value. If the p-value is below 0.05 (5%), we say the pattern is <strong>statistically significant</strong> — there's less than a 5% chance that the relationship is coincidence.<br><br>
  Think of it like a courtroom: the model is on trial, and each coefficient is a piece of evidence. The p-value is the probability that the evidence is fake. Below 0.05 = the evidence holds up. Above 0.05 = reasonable doubt.
</div>


In [ ]:
# 6c. P-values from the SARIMAX model
print("SARIMAX COEFFICIENT P-VALUES")
print("=" * 65)
print(f"{'Parameter':<20} {'Coefficient':>12} {'P-value':>10} {'Significant?':>18}")
print("-" * 65)

for param, coef, pval in zip(
    sarimax_result.param_names,
    sarimax_result.params,
    sarimax_result.pvalues
):
    if param == "sigma2":
        sig = "(variance term)"
    elif pval < 0.001:
        sig = "✅ Yes  (p < 0.001)"
    elif pval < 0.01:
        sig = "✅ Yes  (p < 0.01)"
    elif pval < 0.05:
        sig = "✅ Yes  (p < 0.05)"
    else:
        sig = "❌ No   (p ≥ 0.05)"
    print(f"{param:<20} {coef:>12.4f} {pval:>10.6f} {sig:>18}")

print()
n_sig = sum(1 for p in sarimax_result.pvalues if p < 0.05)
n_total = len(sarimax_result.pvalues)
print(f"Summary: {n_sig} of {n_total} parameters are statistically significant.")


<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #7D6608;">⚠️ P-VALUES — WHAT THE BOARD ASKS</strong><br>
  The board will never ask "what are the p-values?" They'll ask: <strong>"How do you know this forecast isn't just guessing?"</strong><br><br>
  Your answer: <em>"Every pattern our model uses was tested for statistical significance. The month-to-month momentum, the seasonal December peak, the year-over-year growth rate — each one passed a rigorous test showing less than a 5% chance it's random noise. We're not forecasting from gut feel — we're forecasting from patterns that are statistically real."</em>
</div>


### 6d. AIC — Akaike Information Criterion

<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 THE CONCEPT</strong><br>
  AIC answers: <strong>"Is this model the best balance of accuracy vs. simplicity?"</strong><br><br>
  A model with 20 parameters will almost always fit the training data better than a model with 3 — but it will often <em>forecast worse</em> because it memorized noise. AIC penalizes complexity: it rewards good fit and punishes unnecessary parameters.<br><br>
  <strong>Lower AIC = better.</strong> But AIC only means something in comparison — it tells you that the selected model beat all the other combinations <code>auto_arima</code> tested.
</div>


In [ ]:
# 6d. AIC — Why auto_arima chose these parameters
print("AIC — MODEL SELECTION")
print("=" * 50)
print(f"Selected model: SARIMAX{order}x{seasonal_order}")
print(f"AIC: {auto_model.aic():.2f}")
print()
print("AIC = 2k − 2·ln(L)")
print("  k = number of parameters (complexity penalty)")
print("  L = likelihood of the data given the model (fit quality)")
print()
print(f"Note: The AIC is negative ({auto_model.aic():.2f}) because we're working")
print(f"in log-space. Log-transformed data has smaller values, so the likelihood")
print(f"calculation produces negative AIC. This is perfectly normal — lower")
print(f"(more negative) is still better.")
print()
print("BOARD TRANSLATION:")
print('"We tested hundreds of model configurations and selected the')
print(' one that best balances accuracy with simplicity — ensuring')
print(" we're capturing real patterns, not noise.\"")


### 6e. Summary — The Metrics Cheat Sheet

| Metric | What It Measures | Good Value | Bad Value | Board Translation |
|--------|-----------------|------------|-----------|-------------------|
| **RMSE** | Average forecast error in real units | Low relative to data scale | High relative to data scale | "Our forecast is typically within $X of actual" |
| **R²** | % of patterns captured | 0.70–1.00 | Negative (worse than average) | "We capture X% of what drives sales" |
| **P-value** | Is this pattern real or noise? | < 0.05 (significant) | ≥ 0.05 (not significant) | "Every pattern we use passed a statistical reliability test" |
| **AIC** | Model selection score | Lower wins (relative only) | Higher than alternatives | "We tested hundreds of models and chose the best one" |


---
## Block 7: Model Comparison & Board Recommendation (~15 min)


In [ ]:
# 7a. Side-by-side comparison table
print("MODEL COMPARISON — SARIMAX vs PROPHET")
print("=" * 65)
print(f"{'Metric':<30} {'SARIMAX':>15} {'Prophet':>15}")
print("-" * 65)
print(f"{'RMSE ($ millions)':<30} {f'${sarimax_rmse:,.0f}M':>15} {f'${prophet_rmse:,.0f}M':>15}")
print(f"{'RMSE as % of avg sales':<30} {f'{sarimax_rmse/test_actual.mean()*100:.1f}%':>15} {f'{prophet_rmse/test_actual.mean()*100:.1f}%':>15}")
print(f"{'R²':<30} {f'{sarimax_r2:.4f}':>15} {f'{prophet_r2:.4f}':>15}")
print(f"{'R² (interpretation)':<30} {f'{sarimax_r2*100:.1f}% captured':>15} {'Negative ❌':>15}")
print(f"{'Approach':<30} {'Statistical':>15} {'Curve-fitting':>15}")
print(f"{'Log-transform?':<30} {'Yes':>15} {'Yes':>15}")
print("-" * 65)
print(f"\n🏆 SARIMAX wins decisively.")
print(f"   RMSE advantage: ${abs(sarimax_rmse - prophet_rmse):,.0f}M per month")
print(f"   Prophet's negative R² means it should NOT be used for this forecast.")


In [ ]:
# 7b. The Board Chart — Combined overlay (this is what you present)
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(train_actual.index[-60:], train_actual[-60:],
        label="Historical Sales (2018–2022)", color="steelblue", alpha=0.5, linewidth=1.5)
ax.plot(test_actual.index, test_actual,
        label="Actual Sales (2023–2025)", color="darkorange", linewidth=2.5)
ax.plot(test_actual.index, sarimax_forecast,
        label=f"SARIMAX (RMSE=${sarimax_rmse:,.0f}M, R²={sarimax_r2:.2f})",
        linestyle="--", color="green", linewidth=2)
ax.plot(test_actual.index, prophet_forecast,
        label=f"Prophet (RMSE=${prophet_rmse:,.0f}M, R²={prophet_r2:.2f})",
        linestyle="--", color="purple", linewidth=2)

ax.axvline(x=test_actual.index[0], color="red", linestyle="--", alpha=0.3)
ax.text(test_actual.index[0], ax.get_ylim()[1]*0.98, " → Forecast begins ",
        fontsize=9, color="red", alpha=0.7, va="top")

ax.set_title("Board Presentation Chart — SARIMAX vs Prophet\n"
             "U.S. Food Services & Drinking Places: 36-Month Forecast Evaluation",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Month", fontsize=11)
ax.set_ylabel("Monthly Sales (Millions USD)", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:,.0f}B'))
ax.legend(loc="upper left", fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 7c. Error distribution — Where do the models struggle?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(test_actual))

axes[0].bar(x - 0.2, errors_sarimax / 1000, 0.4, label="SARIMAX", color="green", alpha=0.7)
axes[0].bar(x + 0.2, errors_prophet / 1000, 0.4, label="Prophet", color="purple", alpha=0.7)
axes[0].axhline(y=0, color="black", linewidth=0.5)
axes[0].set_xlabel("Test Month")
axes[0].set_ylabel("Error (Billions USD)")
axes[0].set_title("Monthly Forecast Errors\n(+ = Under-predicted, − = Over-predicted)", fontsize=11)
axes[0].set_xticks(x[::6])
axes[0].set_xticklabels([test_actual.index[i].strftime('%b\n%Y') for i in range(0, len(test_actual), 6)], fontsize=8)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(errors_sarimax / 1000, bins=10, alpha=0.6, label="SARIMAX", color="green")
axes[1].hist(errors_prophet / 1000, bins=10, alpha=0.6, label="Prophet", color="purple")
axes[1].axvline(x=0, color="black", linewidth=1)
axes[1].set_xlabel("Forecast Error (Billions USD)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Error Distribution\n(Centered on zero = no bias)", fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"SARIMAX mean error (bias): ${errors_sarimax.mean():+,.0f}M")
print(f"Prophet mean error (bias):  ${errors_prophet.mean():+,.0f}M")
print()
print("Both models have negative bias — they over-predict (forecast > actual).")
print(f"SARIMAX over-predicts by ~${abs(errors_sarimax.mean()):,.0f}M/month on average.")
print(f"Prophet over-predicts by ~${abs(errors_prophet.mean()):,.0f}M/month — much worse.")
print()
print("For the board: A model that over-predicts is 'optimistic' —")
print("it expects more revenue than materializes. Budget conservatively.")


### 7d. Writing the Board Recommendation

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ THE TEMPLATE — What the Board Hears</strong><br>
  A board recommendation has four parts. Practice saying this out loud:
  <ol>
    <li><strong>The Bottom Line:</strong> Which model and why?</li>
    <li><strong>The Evidence:</strong> Two numbers — RMSE in dollars, R² as a percentage.</li>
    <li><strong>The Risk Buffer:</strong> ±X% range for budget planning.</li>
    <li><strong>The Caveat:</strong> What could make this forecast wrong?</li>
  </ol>
</div>


In [ ]:
# 7e. Generate the board recommendation
pct_error = sarimax_rmse / test_actual.mean() * 100
example_month = 700000  # $700M projected month
buffer_low = example_month - sarimax_rmse
buffer_high = example_month + sarimax_rmse

print("=" * 65)
print("BOARD RECOMMENDATION — READY TO PRESENT")
print("=" * 65)
print()
print(f"1. THE BOTTOM LINE:")
print(f'   "We recommend the SARIMAX model for our 2026 monthly')
print(f'    sales forecast. Prophet was also evaluated but did not')
print(f'    meet our accuracy threshold."')
print()
print(f"2. THE EVIDENCE:")
print(f'   "SARIMAX predicts within ${sarimax_rmse:,.0f}M ({pct_error:.1f}%) of')
print(f'    actual monthly sales and captures {sarimax_r2*100:.0f}% of month-to-month')
print(f'    variation — tested against 36 months of actual data the model')
print(f'    had never seen. Prophet was ${prophet_rmse:,.0f}M off on average and')
print(f'    performed worse than a simple historical average."')
print()
print(f"3. THE RISK BUFFER:")
print(f'   "For a projected $700M month, budget for a range of')
print(f'    ${buffer_low:,.0f}M to ${buffer_high:,.0f}M (±{pct_error:.0f}%).')
print(f'    Note: the model has a slight optimistic bias — it tends to')
print(f'    over-predict by ~${abs(errors_sarimax.mean()):,.0f}M, so actual revenue')
print(f'    may come in at the lower end of this range."')
print()
print(f"4. THE CAVEAT:")
print(f'   "This model was trained on 13 years of data including the COVID')
print(f'    recovery. It assumes no comparable disruptions in the forecast')
print(f'    period. The model also assumes the current growth rate will persist —')
print(f'    if consumer spending patterns shift, we recommend quarterly')
print(f'    re-calibration with the latest data."')


---
## Discussion Questions (Remaining Time)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DISCUSS WITH YOUR NEIGHBOR</strong><br>
  <ol>
    <li><strong>The log-transform changed everything.</strong> Without it, SARIMAX had negative R². With it, R² jumped to ~76%. Why does taking the log of sales data help a forecasting model so much? (Hint: think about what the seasonal peaks look like in 1995 vs. 2024.)</li>
    <li><strong>The hotel exercise had negative R² for both models.</strong> Tonight's retail data with log-transform has a strong positive R² for SARIMAX. What's different about the setup? (Hint: training window, data transformation, and test set.)</li>
    <li><strong>You're the CFO.</strong> A data analyst shows you a model with RMSE of $23,000M and R² of 0.76. Is this a good model? What question do you ask before deciding? (Hint: what's the scale of the data?)</li>
    <li><strong>P-value trap:</strong> A model parameter has a p-value of 0.06. The analyst says "it's almost significant, so we should keep it." Do you agree or disagree? Why?</li>
    <li><strong>Prophet lost tonight.</strong> Does that mean Prophet is always worse than SARIMAX? Under what conditions might Prophet win? (Hint: think about the type of data Prophet was designed for.)</li>
    <li><strong>Bias matters.</strong> Our SARIMAX model over-predicts by ~$18,000M/month. If you're the CFO, would you prefer a model that over-predicts (optimistic) or under-predicts (conservative)? Why?</li>
  </ol>
</div>


---
## Key Takeaways from Tonight

1. **RMSE tells the business how much to budget for error** — it's in real dollars, not abstract units
2. **R² tells the analyst how much of the story the model captures** — 0.76 = "76% of patterns explained"
3. **P-values tell the skeptic that the patterns are real** — below 0.05 = statistically reliable
4. **AIC tells the modeler they picked the best configuration** — lower = better balance of fit and simplicity
5. **The board doesn't care about the math** — they care about the dollar range, the confidence level, and the caveat
6. **A negative R² means the model is worse than guessing the average** — that's a model you discard, not present
7. **The log-transform is standard practice for economic time series** — it stabilizes growth and makes seasonal patterns constant-sized, which is exactly what SARIMAX needs
8. **Data preparation matters as much as model selection** — the same SARIMAX model went from unusable (negative R²) to strong (76%) with one line of code: `np.log(ts)`

---
<p style="color:#7F8C8D; font-size:0.85em;">
<em>CAP4767 Data Mining with Python | Miami Dade College</em><br>
Week 2 Demo — Forecasting U.S. Retail Sales: SARIMAX & Prophet with Board-Level Interpretation<br>
Data Source: FRED Series RSAFSNA (Advance Retail Sales: Food Services & Drinking Places)
</p>
